In [15]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
df_merged = pd.read_csv("/content/drive/MyDrive/CRIS/df_merged_full.csv")
pivot = pd.read_csv("/content/drive/MyDrive/CRIS/brand_aspect_rating.csv", index_col=0)
neg_pivot = pd.read_csv("/content/drive/MyDrive/CRIS/brand_negative_rate.csv", index_col=0)
summary = pd.read_csv("/content/drive/MyDrive/CRIS/competitor_summary_table.csv", index_col=0)
brand_df = pd.read_csv("/content/drive/MyDrive/CRIS/brand_df.csv")
negative_df = pd.read_csv("/content/drive/MyDrive/CRIS/negative_brand_reviews.csv")

df_merged.head()

,rating,title,text,images_x,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,average_rating,rating_number,features,description,price,images_y,videos,store,categories,details
0,3.0,Battery life sucks,Battery lasted 2 months ...sucks,[],B085L34V8J,B085L34V8J,AESGKFNIDELMTPT7OLS67JA545JA,1603999669246,0,True,...,4.1,1091,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Gorsun Wireless Bluetooth Sports E...,gorsun,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Package Dimensions': '7.32 x 5.94 x 1.26 inc...
1,1.0,JUNK!!!,These are absolute junk. Not at all the Bose ...,[],B01L7PWBRG,B0B62FJF1J,AEAU4B3HGL46SP3ARQZZ3B7PDA4Q,1489026041000,0,False,...,4.4,39355,['Note : If the size of the earbud tips does n...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Bose,"['Electronics', 'Headphones, Earbuds & Accesso...","{'Brand': 'Bose', 'Model Name': 'SoundSport', ..."
2,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...
3,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...
4,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...


In [17]:
def generate_business_recommendations(neg_pivot, pivot):
    recs = []

    for aspect in neg_pivot.columns:
        worst_brand = neg_pivot[aspect].idxmax()
        worst_rate = neg_pivot[aspect].max()

        best_brand = pivot[aspect].idxmax()
        best_rating = pivot[aspect].max()

        if worst_rate >= 0.05:
            severity = "critical"
        elif worst_rate >= 0.03:
            severity = "high"
        elif worst_rate >= 0.015:
            severity = "moderate"
        else:
            severity = "low"

        rec_text = (
            f"{aspect.capitalize()} is a {severity} issue. "
            f"{worst_brand} has the highest complaint rate at {worst_rate*100:.2f}%, "
            f"while {best_brand} leads this aspect with an average rating of {best_rating:.2f}. "
            f"Teams should prioritize improvements in {aspect} to reduce dissatisfaction and match top competitors."
        )

        recs.append({
            "aspect": aspect,
            "severity": severity,
            "worst_brand": worst_brand,
            "worst_negative_rate_pct": round(worst_rate*100, 2),
            "best_brand": best_brand,
            "best_aspect_rating": round(best_rating, 2),
            "business_recommendation": rec_text
        })

    return pd.DataFrame(recs)

business_recs = generate_business_recommendations(neg_pivot, pivot)
business_recs.sort_values(by="worst_negative_rate_pct", ascending=False)

,aspect,severity,worst_brand,worst_negative_rate_pct,best_brand,best_aspect_rating,business_recommendation
3,comfort,critical,LG,7.91,Audio-Technica,4.49,Comfort is a critical issue. LG has the highes...
2,build,critical,Skullcandy,5.52,Audio-Technica,4.60,Build is a critical issue. Skullcandy has the ...
7,sound,high,Bose,4.97,Audio-Technica,4.61,Sound is a high issue. Bose has the highest co...
5,microphone,high,LG,4.32,Audio-Technica,4.56,Microphone is a high issue. LG has the highest...
1,battery,high,Bose,3.73,Audio-Technica,4.83,Battery is a high issue. Bose has the highest ...
4,connectivity,high,Skullcandy,3.59,Sennheiser,4.44,Connectivity is a high issue. Skullcandy has t...
6,price,moderate,Bose,2.17,Anker,4.62,Price is a moderate issue. Bose has the highes...
0,anc,moderate,Sony,1.92,Anker,5.00,Anc is a moderate issue. Sony has the highest ...


In [10]:
business_recs.to_csv("/content/drive/MyDrive/CRIS/business_recommendations.csv", index=False)

In [18]:
def generate_business_recommendations(neg_pivot, pivot):
    recs = []

    for aspect in neg_pivot.columns:
        worst_brand = neg_pivot[aspect].idxmax()
        worst_rate = neg_pivot[aspect].max()

        best_brand = pivot[aspect].idxmax()
        best_rating = pivot[aspect].max()

        if worst_rate >= 0.05:
            severity = "critical"
        elif worst_rate >= 0.03:
            severity = "high"
        elif worst_rate >= 0.015:
            severity = "moderate"
        else:
            severity = "low"

        rec_text = (
            f"{aspect.capitalize()} is a {severity} issue. "
            f"{worst_brand} has the highest complaint rate at {worst_rate*100:.2f}%, "
            f"while {best_brand} leads this aspect with an average rating of {best_rating:.2f}. "
            f"Teams should prioritize improvements in {aspect} to reduce dissatisfaction and match top competitors."
        )

        recs.append({
            "aspect": aspect,
            "severity": severity,
            "worst_brand": worst_brand,
            "worst_negative_rate_pct": round(worst_rate*100, 2),
            "best_brand": best_brand,
            "best_aspect_rating": round(best_rating, 2),
            "business_recommendation": rec_text
        })

    return pd.DataFrame(recs)

business_recs = generate_business_recommendations(neg_pivot, pivot)
business_recs.sort_values(by="worst_negative_rate_pct", ascending=False)
business_recs.to_csv("/content/drive/MyDrive/CRIS/business_recommendations.csv", index=False)

In [19]:
def generate_brand_summary(brand_df):
    summaries = []

    for brand in sorted(brand_df["store"].unique()):
        subset = brand_df[brand_df["store"] == brand]

        avg_rating = subset["rating"].mean()
        total_reviews = len(subset)

        negative_subset = subset[subset["rating"] <= 2]

        if len(negative_subset) > 0:
            top_issue = negative_subset["aspects"].value_counts().idxmax()
        else:
            top_issue = "No major issue"

        best_aspect = subset.groupby("aspects")["rating"].mean().idxmax()

        summaries.append({
            "brand": brand,
            "total_reviews": total_reviews,
            "avg_rating": round(avg_rating, 2),
            "best_aspect": best_aspect,
            "top_issue": top_issue
        })

    return pd.DataFrame(summaries)


brand_summary_df = generate_brand_summary(brand_df)
brand_summary_df

,brand,total_reviews,avg_rating,best_aspect,top_issue
0,Anker,178,4.23,anc,sound
1,Audio-Technica,308,4.54,battery,comfort
2,Bose,322,3.71,sound,comfort
3,JVC,123,3.80,anc,comfort
4,LG,139,3.73,price,comfort
5,Sennheiser,227,4.36,anc,comfort
6,Skullcandy,362,3.51,microphone,build
7,Sony,938,3.89,battery,sound
8,SoundPEATS,172,3.72,price,comfort
9,Soundcore,132,3.97,sound,comfort


In [20]:
def generate_pm_summary(brand_summary_df):
    rows = []

    for _, row in brand_summary_df.iterrows():
        summary_text = (
            f"{row['brand']} has an average rating of {row['avg_rating']} across "
            f"{row['total_reviews']} reviews. Its strongest feature is {row['best_aspect']}, "
            f"while the most common customer complaint is {row['top_issue']}. "
            f"Improving {row['top_issue']} could significantly enhance customer satisfaction."
        )

        rows.append({
            "brand": row["brand"],
            "pm_summary": summary_text
        })

    return pd.DataFrame(rows)


pm_summary_df = generate_pm_summary(brand_summary_df)
pm_summary_df

,brand,pm_summary
0,Anker,Anker has an average rating of 4.23 across 178...
1,Audio-Technica,Audio-Technica has an average rating of 4.54 a...
2,Bose,Bose has an average rating of 3.71 across 322 ...
3,JVC,JVC has an average rating of 3.8 across 123 re...
4,LG,LG has an average rating of 3.73 across 139 re...
5,Sennheiser,Sennheiser has an average rating of 4.36 acros...
6,Skullcandy,Skullcandy has an average rating of 3.51 acros...
7,Sony,Sony has an average rating of 3.89 across 938 ...
8,SoundPEATS,SoundPEATS has an average rating of 3.72 acros...
9,Soundcore,Soundcore has an average rating of 3.97 across...


In [21]:
pm_summary_df.to_csv("/content/drive/MyDrive/CRIS/pm_summary.csv", index=False)